# ⏰ Notebook 6: Signals & Timers

Waiting for external events and handling long delays.

## Learning Objectives

By the end of this notebook, you'll understand:
- Signals for external events
- Durable timers
- Human-in-the-loop workflows
- Combining signals with timeouts

In [ ]:
import asyncio
from datetime import timedelta
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.worker import Worker
import uuid

print("✅ Temporal SDK imported!")

## 📡 What Are Signals?

In [ ]:
print("📡 Signals = External Events to Workflows")
print("=" * 60)
print("""
PROBLEM: How to notify a workflow of external events?

Examples:
• Payment webhook received
• User approved a document
• Package was picked up
• Admin cancelled an order

POLLING APPROACH (BAD):
─────────────────────────────────────────────────────────────
  while not approved:
      await sleep(30)  # Waste resources!
      status = await check_database()

SIGNAL APPROACH (GOOD):
─────────────────────────────────────────────────────────────
  await workflow.wait_condition(lambda: self.approved)
  # Workflow suspends! Uses zero resources!
  # External system sends signal when ready.

HOW IT WORKS:
─────────────────────────────────────────────────────────────
                                    
  [Workflow]                        [External System]
      │                                   │
      ▼                                   │
  wait_condition() ◄──── signal ──────────┘
      │                  "approved!"
      ▼
  continue execution

Signals are:
• Persisted in workflow history
• Delivered exactly once
• Can carry data
""")

## 🔧 Signal Example: Document Approval

In [ ]:
from dataclasses import dataclass

@dataclass
class DocumentInput:
    doc_id: str
    title: str
    requester: str

@dataclass
class ApprovalSignal:
    approved: bool
    approver: str
    comments: str = ""

@activity.defn
async def send_approval_request(doc: DocumentInput) -> str:
    print(f"   📧 Sending approval request for '{doc.title}' to reviewers...")
    await asyncio.sleep(0.3)
    return f"request_{doc.doc_id}"

@activity.defn
async def process_approved_document(doc_id: str, approver: str) -> str:
    print(f"   ✅ Processing approved document by {approver}...")
    await asyncio.sleep(0.3)
    return f"processed_{doc_id}"

@activity.defn
async def send_rejection_notice(doc_id: str, reason: str) -> bool:
    print(f"   ❌ Sending rejection notice: {reason}")
    await asyncio.sleep(0.2)
    return True

@workflow.defn
class DocumentApprovalWorkflow:
    def __init__(self):
        self.approval_received = False
        self.approval_result: ApprovalSignal = None
    
    @workflow.signal
    async def submit_approval(self, result: ApprovalSignal):
        print(f"   📡 Signal received: approved={result.approved}")
        self.approval_result = result
        self.approval_received = True
    
    @workflow.run
    async def run(self, doc: DocumentInput) -> dict:
        print(f"\n🚀 Starting approval workflow for '{doc.title}'")
        
        await workflow.execute_activity(
            send_approval_request, doc,
            start_to_close_timeout=timedelta(seconds=30)
        )
        
        print(f"   ⏳ Waiting for approval (up to 5 seconds for demo)...")
        
        try:
            await workflow.wait_condition(
                lambda: self.approval_received,
                timeout=timedelta(seconds=5)
            )
        except asyncio.TimeoutError:
            print(f"   ⏰ Approval timed out!")
            return {"status": "timed_out", "doc_id": doc.doc_id}
        
        if self.approval_result.approved:
            result = await workflow.execute_activity(
                process_approved_document,
                args=[doc.doc_id, self.approval_result.approver],
                start_to_close_timeout=timedelta(seconds=30)
            )
            return {
                "status": "approved",
                "doc_id": doc.doc_id,
                "approver": self.approval_result.approver,
                "result": result
            }
        else:
            await workflow.execute_activity(
                send_rejection_notice,
                args=[doc.doc_id, self.approval_result.comments],
                start_to_close_timeout=timedelta(seconds=30)
            )
            return {
                "status": "rejected",
                "doc_id": doc.doc_id,
                "reason": self.approval_result.comments
            }

print("✅ Document approval workflow defined!")

In [ ]:
async def run_approval_demo():
    client = await Client.connect("localhost:7233")
    print("✅ Connected to Temporal!")
    
    async with Worker(
        client,
        task_queue="approval-queue",
        workflows=[DocumentApprovalWorkflow],
        activities=[send_approval_request, process_approved_document, send_rejection_notice]
    ):
        print("\n👷 Worker started!")
        
        doc = DocumentInput(
            doc_id=str(uuid.uuid4())[:8],
            title="Q4 Budget Proposal",
            requester="alice@example.com"
        )
        
        workflow_id = f"approval-{doc.doc_id}"
        handle = await client.start_workflow(
            DocumentApprovalWorkflow.run,
            doc,
            id=workflow_id,
            task_queue="approval-queue"
        )
        print(f"\n📄 Workflow started: {workflow_id}")
        
        await asyncio.sleep(2)
        
        print("\n📡 Sending approval signal...")
        await handle.signal(
            DocumentApprovalWorkflow.submit_approval,
            ApprovalSignal(
                approved=True,
                approver="bob@example.com",
                comments="Looks good!"
            )
        )
        
        result = await handle.result()
        return result

print("📡 Signal Demo: Document Approval")
print("=" * 60)

try:
    result = await run_approval_demo()
    print(f"\n📊 Final Result:")
    for k, v in result.items():
        print(f"   {k}: {v}")
except Exception as e:
    print(f"\n❌ Error: {e}")

## ⏰ Durable Timers

In [ ]:
print("⏰ Durable Timers")
print("=" * 60)
print("""
PROBLEM: Need to wait for a long time (hours, days, weeks)

NORMAL SLEEP (BAD):
─────────────────────────────────────────────────────────────
  time.sleep(86400)  # Sleep for 24 hours
  
  Problems:
  • Worker can't do other work
  • If worker crashes, timer is lost!
  • Memory wasted holding connection

DURABLE TIMER (GOOD):
─────────────────────────────────────────────────────────────
  await workflow.sleep(timedelta(days=1))
  
  Benefits:
  • Workflow suspends (no resources used)
  • Timer persisted in Temporal server
  • If worker crashes, timer continues!
  • Can wake up on any worker

USE CASES:
─────────────────────────────────────────────────────────────
• Send reminder after 7 days if no response
• Expire offer after 24 hours
• Trial period ends after 30 days
• Scheduled tasks at specific times
""")

In [ ]:
@dataclass
class TrialInput:
    user_id: str
    email: str
    trial_days: int

@activity.defn
async def activate_trial(user_id: str) -> str:
    print(f"   🎉 Activating trial for user {user_id}...")
    await asyncio.sleep(0.2)
    return f"trial_{user_id}"

@activity.defn
async def send_trial_reminder(email: str, days_left: int) -> bool:
    print(f"   📧 Sending reminder to {email}: {days_left} days left!")
    await asyncio.sleep(0.2)
    return True

@activity.defn
async def expire_trial(user_id: str) -> bool:
    print(f"   ⏰ Trial expired for user {user_id}")
    await asyncio.sleep(0.2)
    return True

@activity.defn
async def convert_to_paid(user_id: str) -> str:
    print(f"   💰 Converting user {user_id} to paid!")
    await asyncio.sleep(0.2)
    return f"paid_{user_id}"

@workflow.defn
class TrialWorkflow:
    def __init__(self):
        self.converted = False
    
    @workflow.signal
    async def user_upgraded(self):
        print(f"   📡 User upgraded signal received!")
        self.converted = True
    
    @workflow.run
    async def run(self, input: TrialInput) -> dict:
        print(f"\n🚀 Starting trial workflow for {input.email}")
        
        await workflow.execute_activity(
            activate_trial, input.user_id,
            start_to_close_timeout=timedelta(seconds=30)
        )
        
        trial_seconds = input.trial_days
        reminder_at = trial_seconds // 2
        
        print(f"   ⏳ Waiting {reminder_at}s until reminder (simulating {input.trial_days//2} days)...")
        
        try:
            await workflow.wait_condition(
                lambda: self.converted,
                timeout=timedelta(seconds=reminder_at)
            )
        except asyncio.TimeoutError:
            pass
        
        if self.converted:
            result = await workflow.execute_activity(
                convert_to_paid, input.user_id,
                start_to_close_timeout=timedelta(seconds=30)
            )
            return {"status": "converted_early", "subscription": result}
        
        await workflow.execute_activity(
            send_trial_reminder, 
            args=[input.email, input.trial_days // 2],
            start_to_close_timeout=timedelta(seconds=30)
        )
        
        print(f"   ⏳ Waiting {trial_seconds - reminder_at}s until expiry...")
        
        try:
            await workflow.wait_condition(
                lambda: self.converted,
                timeout=timedelta(seconds=trial_seconds - reminder_at)
            )
        except asyncio.TimeoutError:
            pass
        
        if self.converted:
            result = await workflow.execute_activity(
                convert_to_paid, input.user_id,
                start_to_close_timeout=timedelta(seconds=30)
            )
            return {"status": "converted", "subscription": result}
        else:
            await workflow.execute_activity(
                expire_trial, input.user_id,
                start_to_close_timeout=timedelta(seconds=30)
            )
            return {"status": "expired", "user_id": input.user_id}

print("✅ Trial workflow defined!")

In [ ]:
async def run_trial_demo():
    client = await Client.connect("localhost:7233")
    print("✅ Connected to Temporal!")
    
    async with Worker(
        client,
        task_queue="trial-queue",
        workflows=[TrialWorkflow],
        activities=[activate_trial, send_trial_reminder, expire_trial, convert_to_paid]
    ):
        print("\n👷 Worker started!")
        
        trial_input = TrialInput(
            user_id=str(uuid.uuid4())[:8],
            email="newuser@example.com",
            trial_days=6
        )
        
        workflow_id = f"trial-{trial_input.user_id}"
        handle = await client.start_workflow(
            TrialWorkflow.run,
            trial_input,
            id=workflow_id,
            task_queue="trial-queue"
        )
        
        result = await handle.result()
        return result

print("⏰ Timer Demo: Trial Workflow (will expire)")
print("=" * 60)

try:
    result = await run_trial_demo()
    print(f"\n📊 Final Result:")
    for k, v in result.items():
        print(f"   {k}: {v}")
except Exception as e:
    print(f"\n❌ Error: {e}")

In [ ]:
async def run_trial_with_conversion():
    client = await Client.connect("localhost:7233")
    
    async with Worker(
        client,
        task_queue="trial-queue",
        workflows=[TrialWorkflow],
        activities=[activate_trial, send_trial_reminder, expire_trial, convert_to_paid]
    ):
        trial_input = TrialInput(
            user_id=str(uuid.uuid4())[:8],
            email="eager@example.com",
            trial_days=10
        )
        
        workflow_id = f"trial-{trial_input.user_id}"
        handle = await client.start_workflow(
            TrialWorkflow.run,
            trial_input,
            id=workflow_id,
            task_queue="trial-queue"
        )
        
        print(f"\n📄 Trial started, waiting 2s then upgrading...")
        await asyncio.sleep(2)
        
        print("📡 Sending upgrade signal!")
        await handle.signal(TrialWorkflow.user_upgraded)
        
        result = await handle.result()
        return result

print("\n" + "=" * 60)
print("⏰ Timer Demo: Trial with Early Conversion")
print("=" * 60)

try:
    result = await run_trial_with_conversion()
    print(f"\n📊 Final Result:")
    for k, v in result.items():
        print(f"   {k}: {v}")
    print("\n✅ Timer was cancelled because user converted!")
except Exception as e:
    print(f"\n❌ Error: {e}")

## 🧪 Quick Quiz

1. **What's the difference between signals and activities?**

2. **Why are durable timers better than time.sleep()?**

3. **What happens to a timer if the worker crashes?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Signals vs Activities:")
print("   - Signals: external events TO workflow")
print("   - Activities: work done BY workflow")
print("   - Signals are push, activities are pull")
print()
print("2. Why durable timers:")
print("   - Persisted in Temporal server")
print("   - Survive worker crashes")
print("   - No resources used while waiting")
print()
print("3. Timer during crash:")
print("   - Timer continues on Temporal server")
print("   - When it fires, workflow wakes up")
print("   - Any available worker handles it")

## 📚 Pattern Summary

### What We Learned

| Notebook | Topic | Key Concept |
|----------|-------|-------------|
| 01 | The Problem | Why distributed workflows are hard |
| 02 | Naive Approach | Manual state management pain |
| 03 | Event Sourcing | Events for recovery (still complex) |
| 04 | Temporal Basics | Workflows + Activities |
| 05 | Failures | Heartbeats, replay, recovery |
| 06 | Signals & Timers | External events, durable waits |

### When to Use Workflows

✅ **Use Temporal When:**
- Multiple services need coordination
- Process can fail and needs recovery
- Long waits (human tasks, webhooks)
- Need audit trail of all steps
- Complex compensation logic

❌ **Don't Use When:**
- Simple CRUD operations
- Single service, single step
- Sub-second latency required
- Very high frequency, low value

### Architecture Recap

```
┌────────────────────────────────────────────────────────────┐
│                    Temporal Architecture                    │
├────────────────────────────────────────────────────────────┤
│                                                            │
│  [Your App] ──start──► [Temporal Server] ◄── [Workers]    │
│      │                        │                 │          │
│      │                        ▼                 │          │
│      │                  [PostgreSQL]            │          │
│      │                  (History DB)            │          │
│      │                        │                 │          │
│      └────signal─────────────►│◄───activities───┘          │
│                               │                            │
│                          [Web UI]                          │
│                        localhost:8080                      │
│                                                            │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
print("🎉 Congratulations!")
print("=" * 60)
print("""
You've completed the Multi-Step Processes pattern!

Key takeaways:
✓ Distributed workflows are HARD
✓ Manual state management doesn't scale
✓ Temporal handles the hard parts
✓ Workflows = orchestration logic
✓ Activities = actual work
✓ Heartbeats detect failures quickly
✓ Signals for external events
✓ Timers survive crashes

Next steps:
• Build a real workflow for your domain
• Explore Temporal's advanced features
• Learn about workflow versioning
• Study saga patterns for compensation

Check out: https://temporal.io/docs
""")